In [ ]:
import json

import google.auth
from google.cloud import pubsub_v1
from google.cloud.pubsub_v1.types import ReceivedMessage, PullResponse

In [ ]:

# authenticate as yourself using Application Default Credentials (ADC)
credentials, adc_project = google.auth.default()
PROJECT_ID = 'rmr-cloud-services'
subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
print('ADC project:', adc_project)
print('Credential type:', type(credentials).__name__)

# you can also authenticate using a service account

In [ ]:
# OPTIONAL

# list off every subscription in our project ('rmr-cloud-services')
project_path = f'projects/{PROJECT_ID}'
for sub in subscriber.list_subscriptions(request={'project': project_path}):
    print(sub.name, '->', sub.topic)

In [ ]:

# set pubsub subscription details
SUBSCRIPTION_ID = 'filescom-dlq-sub'
subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)

In [ ]:

# NOTE this is a one-time pull that must be called manually.
# Once we move this code to gateway-services, we will need to implement a method that actively
# listens to the subscription and handles the messages

# method to pull messages from the subscription
def get_messages(
    subscription: str = subscription_path,
    max_messages: int = 50,
    timeout: int = 30
) -> list[ReceivedMessage]:

    # make the request
    response = subscriber.pull(
        request={
            "subscription": subscription,
            "max_messages": max_messages,
        },
        timeout=timeout,
    )

    messages = response.received_messages

    # NOTE: we're not calling subscriber.acknowledge(...) here, so these 
    # messages will redeliver after the ack deadline

    # # to remove the messages after pulling them:
    # ack_ids = [rm.ack_id for rm in messages]
    # if ack_ids:
    #     subscriber.acknowledge(request={'subscription': subscription, 'ack_ids': ack_ids})
    #     num_ack_ids = len(ack_ids)
    #     print(f"Acknowledged {num_ack_ids} message{'s' if num_ack_ids > 1 else ''}")

    num_messages = len(messages)
    print(f"Got {num_messages} message{'s' if num_messages > 1 else ''}")
    return messages


In [ ]:
messages = get_messages()

In [ ]:
json.loads(messages[0].message.data)

In [ ]:
messages[0]